# C02w0 — Complete the missing 27 final texts
**Run this notebook by itself in a fresh kernel, from top to bottom.** Save it under `experiments/C02w0_complete_missing27.ipynb`.

This notebook completes only the missing 27 downstream outputs, using existing C02s `ratio_005/state.json` generations. It does not fit a detector or rerun generation. The existing 27 hyphen-repaired final texts remain unchanged.

Stages: original S/W/C → expanded/lower-bar WPS repair where needed → gated punctuation → exclamation → hyphen repair. Reference-policy extensions are documented below. `complete` means every stage ran to its stopping rule, not that every target was reached or quality reviewed.

Each stage checkpoints independently. If interrupted, rerun from the top; completed stages are reused. GPT-2 is the only transformer loaded.

Writes:
- Completion checkpoints and audits: `results/causal/C02w0_complete_missing27_v2/`
- The full 54-text input expected by C02w: `results/causal/C02w_adaptive_camouflage_v1/final_texts_54.csv`

After completion, return to your adaptation notebook and rerun its first two code cells, then continue. No manual copying of texts or status edits is needed.

In [ ]:
# Export every figure to reproduction/figures as well as the run directory.
from pathlib import Path as _FigurePath
def _export_figure(figure, filename, *args, **kwargs):
    original = _FigurePath(filename)
    original.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(original, *args, **kwargs)
    figure_dir = ROOT / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    try:
        suffix = original.relative_to(OUT).with_suffix("").as_posix().replace("/", "_")
    except ValueError:
        suffix = original.stem
    target = figure_dir / ("C02w0_" + suffix + original.suffix)
    figure.savefig(target, *args, **kwargs)

from pathlib import Path
import json,hashlib,os,pickle,re
from collections import Counter
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy,textstat,sklearn
from sklearn.base import clone
from sklearn.metrics import roc_auc_score,roc_curve,auc
from IPython.display import display
ROOT=Path.cwd() if (Path.cwd()/'results').exists() else Path.cwd().parent
assert (ROOT/'results').exists()
BASE=ROOT/'results/causal/C02d2_fz118_ghostbuster_raid'
PRIOR=ROOT/'results/causal/C02e2_raid_feature_guided'
PARENT=ROOT/'results/causal/C02r_discovery_gated_punctuation'
PREV=ROOT/'results/causal/C02s_relative_likelihood_sensitivity'
LATEST=PREV/'wps_repair_v1/downstream_v1/hyphen_repair_v1/texts.csv'
OUT=ROOT/'results/causal/C02w_adaptive_camouflage_v1';OUT.mkdir(exist_ok=True,parents=True)
SEED=20260909;B=2000;BUDGETS=[.01,.05,.10,.20]
normalize=lambda t:str(t).replace('\n',' ').replace('\r',' ')
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def atomic(p,obj):
    p=Path(p);tmp=p.with_suffix('.tmp');tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False));os.replace(tmp,p)
def freeze(p,obj):
    if p.exists():assert json.loads(p.read_text())==obj,f'Changed contract: use new output version: {p}'
    else:atomic(p,obj)
selection=pd.read_csv(PARENT/'selection.csv')
selection=selection.loc[selection.partition.eq('discovery')].sort_values('case_id').reset_index(drop=True)
assert len(selection)==54 and selection.case_id.is_unique and selection.source_id.is_unique
# Largest-remainder genre allocation; fixed seed shuffles only within genre.
counts=selection.domain.value_counts().sort_index();quota=counts*20/54
allocation=np.floor(quota).astype(int)
for g in sorted(counts.index,key=lambda g:(-(quota[g]-allocation[g]),g))[:20-int(allocation.sum())]:allocation[g]+=1
rng=np.random.default_rng(SEED);adapt=[]
for g in counts.index:
    ids=selection.loc[selection.domain.eq(g),'case_id'].to_numpy().copy();rng.shuffle(ids);adapt.extend(ids[:allocation[g]])
selection['role']=np.where(selection.case_id.isin(adapt),'adaptation','test')
assert sum(selection.role.eq('adaptation'))==20 and sum(selection.role.eq('test'))==34
freeze(OUT/'split.json',dict(seed=SEED,selection_sha=sha(PARENT/'selection.csv'),rows=selection[['case_id','source_id','domain','role']].to_dict('records')))
selection.to_csv(OUT/'split.csv',index=False)
latest=pd.read_csv(LATEST,keep_default_na=False);assert latest.case_id.is_unique
assert set(latest.case_id)<=set(selection.case_id)
assert len(latest)==27 and latest.text.str.strip().ne('').all()
known=latest.set_index('case_id').text.to_dict();coverage=[]
for r in selection.itertuples():
    gp=PREV/r.case_id/'ratio_005/state.json'
    state=json.loads(gp.read_text())['state'] if gp.exists() else {}
    coverage.append(dict(case_id=r.case_id,domain=r.domain,role=r.role,
        controlled_status=state.get('status','missing'),full_pipeline_available=r.case_id in known))
coverage=pd.DataFrame(coverage);coverage.to_csv(OUT/'coverage.csv',index=False);display(coverage)
INPUT=OUT/'final_texts_54.csv'
if not INPUT.exists():
    template=selection[['case_id','source_id','domain']].copy()
    template['text']=template.case_id.map(known).fillna('')
    template['pipeline_status']=np.where(template.case_id.isin(known),'complete','missing')
    template['provenance']=np.where(template.case_id.isin(known),str(LATEST.relative_to(ROOT)),'')
    template.to_csv(INPUT,index=False)
print('Saved split and coverage. Fully processed:',len(known),'of 54.')
print('Input template:',INPUT)

existing_input=pd.read_csv(INPUT,keep_default_na=False)
assert existing_input.case_id.is_unique and set(existing_input.case_id)==set(selection.case_id)
for cid,t in known.items():
    assert existing_input.set_index('case_id').loc[cid,'text']==t,f'Existing final text differs: {cid}'
print('This notebook completes the 27 missing downstream outputs; it has no detector-fitting cells.')


## Complete missing final texts from existing 0.05 generations
This section requires GPT-2 scoring, but **does not regenerate any AI text**. Existing 27 final outputs are untouched. Missing cases receive the original sentence pass, expanded/lower-bar WPS continuation when needed, gated punctuation, exclamation treatment and hyphen repair.

**Explicit reference-policy extension for this expanded cohort:** retain saved parent reference IDs and bounds whenever available, even when the old common-arm eligibility test failed; record current length-window support. For cases without a parent pool, choose up to 30 same-genre training humans closest in log word count, minimum 15. These new pools are not constrained to the old 1.25 length window; all exceptions are logged. This is an extension of coverage, not a claim that these cases met the old protocol. Never select on detector outcomes or split role. All generated candidates remain training-reference-driven. Report reference-policy subgroups with the final experiment.

Case-stage checkpoints make interruptions resumable. `complete` means the algorithm ran to its stopping condition, not that all targets were attained or fidelity reviewed.

In [ ]:
import math,copy,gc,torch
from transformers import AutoTokenizer,AutoModelForCausalLM
COMPLETION=ROOT/'results/causal/C02w0_complete_missing27_v2';COMPLETION.mkdir(exist_ok=True,parents=True)
parent_treatment=json.loads((PARENT/'treatment_contract.json').read_text())
parent_refs={r['case_id']:r for r in parent_treatment['prepared']}
features=pd.read_parquet(BASE/'fz118_paired_features.parquet')
corpus=pd.read_parquet(BASE/'paired_corpus.parquet')
assert corpus.source_id.is_unique and features.source_id.is_unique
text_map=corpus.set_index('source_id').text.to_dict()
humans=features.loc[features.dataset.eq('raid')&features.split.eq('train')&features.label.eq(0)].copy()
nlp=spacy.load('en_core_web_sm')
SELECTION_SEED=20260906;BOOTSTRAPS=3000;ARTIFICIAL_SD_SCALE=.5;BUMP_P_MIN=.10
MEAN_DISTANCE_TOLERANCE={'wps_mean':.5,'cps_mean':3.}
SENT_METRICS=['wps_mean','cps_mean']
PMARKS={'semicolon':';','colon':':','hyphen':'-','en_dash':'–','em_dash':'—','comma':','}
DENOMS=['relative','per_sentence']
gate=json.loads((PARENT/'frozen_punctuation_gate.json').read_text());active=gate['active_denominators']
assert active.get('hyphen'), 'Expected frozen hyphen gate'
SCORER_ID=parent_treatment['scorer'];SCORER_REVISION=parent_treatment['scorer_revision']
assert SCORER_REVISION and SCORER_REVISION!='main'
LEFT_WORDS=48;RIGHT_WORDS=32
def measure(doc):
    sentences = [s for s in doc.sents if any(t.is_alpha for t in s)]
    tokens = [t for t in doc if t.is_alpha]
    seqs = {
        'wps': [sum(t.is_alpha for t in s) for s in sentences],
        'cps': [len(s.text.strip()) for s in sentences],
        'cpw': [len(t.text) for t in tokens],
    }
    W, S, C = len(tokens), len(sentences), len(doc.text)
    result = {'W':W, 'S':S, 'C':C,
              'detector_C_over_W': C/W if W else np.nan,
              'detector_C_over_S': C/S if S else np.nan,
              'share_unique_words': len({t.text.lower() for t in tokens})/W if W else np.nan}
    for name, seq in seqs.items():
        a = np.asarray(seq, dtype=float)
        result[name+'_mean'] = float(a.mean()) if len(a) else np.nan
        result[name+'_var'] = float(a.var(ddof=0)) if len(a) else np.nan
    return result, seqs

@lru_cache(maxsize=8192)
def stats_for(text): return measure(nlp(text))[0]
human_stats={sid:stats_for(normalize(text_map[sid])) for sid in humans.source_id}
# Cohort-completion inputs are frozen even on resume; do not key on mutable output CSV.
paths=[PARENT/'treatment_contract.json',PARENT/'frozen_punctuation_gate.json',
       BASE/'paired_corpus.parquet',BASE/'fz118_paired_features.parquet',LATEST]
paths += [PREV/cid/'ratio_005/state.json' for cid in selection.case_id]
nb_path=ROOT/'experiments/C02w0_complete_missing27.ipynb'
completion_code=hashlib.sha256(json.dumps([c['source'] for c in json.loads(nb_path.read_text())['cells'] if c['cell_type']=='code']).encode()).hexdigest()
completion_contract=dict(policy='retain_parent_else_nearest30_same_genre_train_min15_logW_v1',
    code=completion_code,inputs={str(p.relative_to(ROOT)):sha(p) for p in paths},
    spacy=nlp.meta,torch=torch.__version__,scorer=SCORER_ID,revision=SCORER_REVISION,
    selection_seed=SELECTION_SEED,bootstraps=BOOTSTRAPS)
freeze(COMPLETION/'contract.json',completion_contract)
CONTRACT_HASH=hashlib.sha256(json.dumps(completion_contract,sort_keys=True).encode()).hexdigest()
# Prepare every historically missing case, independent of current resume progress and split role.
prepared=[];pool_audit=[]
for row in selection.itertuples():
    if row.case_id in known:continue
    payload=json.loads((PREV/row.case_id/'ratio_005/state.json').read_text())
    state=payload['state']
    assert state['status']=='completed_eos',f'{row.case_id}: no completed 0.05 generation; do not substitute another arm'
    text=normalize(state['committed_text']);W=stats_for(text)['W'];assert W>0
    if row.case_id in parent_refs:
        ref=parent_refs[row.case_id];ids=ref['reference_ids'];bounds=ref['bounds'];policy='retained_parent'
    else:
        candidates=[sid for sid in humans.loc[humans.domain.eq(row.domain),'source_id'] if human_stats[sid]['W']>0 and human_stats[sid]['S']>0]
        candidates.sort(key=lambda sid:(abs(math.log(human_stats[sid]['W']/W)),str(sid)))
        ids=candidates[:30];assert len(ids)>=15,f'{row.case_id}: fewer than 15 same-genre training references'
        bounds={m:dict(zip(['low','high'],map(float,np.quantile([human_stats[sid][m] for sid in ids],[.2,.6])))) for m in SENT_METRICS}
        policy='new_nearest_length_pool'
    assert len(ids)>=15 and set(ids)<=set(humans.loc[humans.domain.eq(row.domain),'source_id'])
    support=sum(W/1.25<=human_stats[sid]['W']<=W*1.25 for sid in ids)
    prepared.append(dict(case_id=row.case_id,source_id=row.source_id,domain=row.domain,text=text,reference_ids=ids,bounds=bounds))
    pool_audit.append(dict(case_id=row.case_id,policy=policy,n_refs=len(ids),within_1_25_window=support,
        all_within_window=support==len(ids),reference_ids=json.dumps(ids),bounds=json.dumps(bounds)))
pd.DataFrame(pool_audit).to_csv(COMPLETION/'reference_policy.csv',index=False)
display(pd.DataFrame(pool_audit).drop(columns=['reference_ids','bounds']))


In [ ]:
tokenizer=AutoTokenizer.from_pretrained(SCORER_ID,revision=SCORER_REVISION,use_fast=True)
model=AutoModelForCausalLM.from_pretrained(SCORER_ID,revision=SCORER_REVISION)
device='cuda' if torch.cuda.is_available() else 'cpu';model.to(device).eval()
# Score at most the model context; reject overflow rather than silently truncating.
@torch.inference_mode()
def continuation_score(left,tail):
    enc=tokenizer(left+tail,return_offsets_mapping=True,return_tensors='pt',add_special_tokens=False)
    offsets=enc.pop('offset_mapping')[0].tolist();ids=enc['input_ids'].to(device)
    if ids.shape[1]<2 or ids.shape[1]>model.config.max_position_embeddings:return None
    logp=model(input_ids=ids).logits[0,:-1].log_softmax(-1)
    chosen=logp.gather(1,ids[0,1:,None]).squeeze(1)
    mask=torch.tensor([end>len(left) for start,end in offsets[1:]],device=device)
    return float(chosen[mask].mean()) if mask.any() else None

def score_edit(text,e):
    a,b,replacement=e['start'],e['end'],e['replacement']
    left_parts=list(re.finditer(r'\S+',text[:a]));right_parts=list(re.finditer(r'\S+',text[b:]))
    left=text[left_parts[-LEFT_WORDS].start() if len(left_parts)>=LEFT_WORDS else 0:a]
    right=text[b:b+(right_parts[RIGHT_WORDS-1].end() if len(right_parts)>=RIGHT_WORDS else len(text)-b)]
    # Fixed flanks shared by both alternatives; future text contributes to each score.
    before=continuation_score(left,text[a:b]+right);after=continuation_score(left,replacement+right)
    if before is None or after is None:return None
    return dict(mutability=math.exp(min(0.,after-before)),score_before=before,score_after=after)


def bar(q): return .05+.90/(1+math.exp(float(np.clip(4*(q-1),-60,60))))

def independent(text):
    d=nlp(text.strip())
    return any(t.dep_=='ROOT' and t.pos_ in {'VERB','AUX'} and
               any(c.dep_ in {'nsubj','nsubjpass','csubj','expl'} for c in t.children) for t in d)

def signature(text):
    return [(t.text.lower(),len(t.text)) for t in nlp.make_doc(text) if t.is_alpha]

def apply_ops(text,ops):
    for a,b,r in sorted(ops,reverse=True):text=text[:a]+r+text[b:]
    return text

def proposals(text):
    doc=nlp(text); sentences=[s for s in doc.sents if any(t.is_alpha for t in s)]
    splits=[]; merges=[]
    for s in sentences:
        for t in s:
            if t.text not in {';',','}:continue
            # Every comma joining two independent clauses is considered.
            left=text[s.start_char:t.idx];right=text[t.idx+1:s.end_char]
            if independent(left) and independent(right):
                # Capitalize the first alphabetic character after the new boundary.
                match=re.search(r'[A-Za-z]',right)
                if match:
                    j=t.idx+1+match.start()
                    splits.append([(t.idx,t.idx+1,'.'),(j,j+1,text[j].upper())])
    for a,b in zip(sentences,sentences[1:]):
        # Restrict merging to plain periods between two independent clauses.
        end=a.end_char
        if text[end-1:end]!='.' or text[end:b.start_char].strip():continue
        if independent(a.text[:-1]) and independent(b.text):
            merges.append([(end-1,end,';')])
    out=[('split',o) for o in splits]+[('merge',o) for o in merges]
    for a in splits:
        for b in merges:
            if all(x[1]<=y[0] or y[1]<=x[0] for x in a for y in b):
                out.append(('boundary_shift',a+b))
    seen=set()
    for rule,ops in out:
        candidate=apply_ops(text,ops)
        if candidate not in seen:
            seen.add(candidate);yield rule,ops,candidate

def distances(stats,b):
    return np.array([max(b[m]['low']-stats[m],0,stats[m]-b[m]['high']) /
        max(b[m]['high']-b[m]['low'],1.0) for m in SENT_METRICS])

def run_case(c):
    text=c['text']; audit=[]; initial_sig=signature(text)
    stats,_=measure(nlp(text));initial=stats.copy(); seen={text}
    while True:
        before_d=distances(stats,c['bounds']); loss=float(before_d.sum())
        if loss<=1e-12: status='in_band';break
        threshold=bar(float(before_d.max()))
        accepted=[]; checked=0
        for rule,ops,candidate in proposals(text):
            checked+=1
            if candidate in seen or signature(candidate)!=initial_sig:continue
            after,_=measure(nlp(candidate))
            expected_delta={'split':1,'merge':-1,'boundary_shift':0}[rule]
            if after['S']!=stats['S']+expected_delta or after['S']<2:continue
            d=distances(after,c['bounds'])
            if d.sum()>=loss-1e-10:continue
            a=min(o[0] for o in ops); b=max(o[1] for o in ops)
            replacement=apply_ops(text[a:b],[(x-a,y-a,r) for x,y,r in ops])
            score=score_edit(text,dict(start=a,end=b,replacement=replacement))
            if score is None or score['mutability']<threshold:continue
            accepted.append((score['mutability'],loss-float(d.sum()),rule,ops,candidate,after,score))
        if not accepted:status='no_eligible_improving_candidate';break
        accepted.sort(key=lambda x:(-x[0],-x[1],x[2],x[4]))
        mut,gain,rule,ops,candidate,after,score=accepted[0]
        audit.append(dict(step=len(audit)+1,rule=rule,ops=ops,before=stats,after=after,
            objective_before=loss,objective_after=loss-gain,threshold=threshold,**score))
        text=candidate;stats=after;seen.add(text)
        print(c['case_id'],len(audit),rule,'loss=',round(loss-gain,4),flush=True)
    assert signature(text)==initial_sig
    for m in ['cpw_mean','cpw_var','share_unique_words']:
        assert np.isclose(stats[m],initial[m],atol=1e-12)
    return dict(text=text,status=status,audit=audit,initial=initial,final=stats,contract_hash=CONTRACT_HASH)
_wps_original_proposals=proposals
_wps_original_bar=bar
def imperative_clause(text):
    """Conservative heuristic for subjectless instructions, e.g. 'Stir well'."""
    doc = nlp(text.strip())
    roots = [t for t in doc if t.dep_ == "ROOT"]
    if len(roots) != 1:
        return False
    root = roots[0]
    if root.pos_ != "VERB" or root.tag_ != "VB":
        return False
    if any(t.dep_ in {"nsubj", "nsubjpass", "csubj", "expl"} for t in root.children):
        return False
    # Avoid treating subordinate clauses as standalone instructions.
    if any(t.dep_ == "mark" for t in root.children):
        return False
    return True

def standalone_clause(text):
    return independent(text) or imperative_clause(text)

def expanded_proposals(text):
    # Preserve every original candidate.
    seen = set()
    for rule, ops, candidate in _wps_original_proposals(text):
        seen.add(candidate)
        yield rule, ops, candidate

    doc = nlp(text)
    sentences = [s for s in doc.sents if any(t.is_alpha for t in s)]
    added = []

    for sentence in sentences:
        for token in sentence:
            if token.text not in {",", ";"}:
                continue
            left = text[sentence.start_char:token.idx]
            right = text[token.idx + 1:sentence.end_char]
            if not (standalone_clause(left) and standalone_clause(right)):
                continue
            match = re.search(r"[A-Za-z]", right)
            if match:
                j = token.idx + 1 + match.start()
                added.append(("split", [
                    (token.idx, token.idx + 1, "."),
                    (j, j + 1, text[j].upper()),
                ]))

    for left, right in zip(sentences, sentences[1:]):
        end = left.end_char
        if text[end - 1:end] != ".":
            continue
        # Preserve paragraph boundaries.
        gap = text[end:right.start_char]
        if gap.strip() or "\n" in gap:
            continue
        if standalone_clause(left.text[:-1]) and standalone_clause(right.text):
            added.append(("merge", [(end - 1, end, ";")]))

    for rule, ops in added:
        candidate = apply_ops(text, ops)
        if candidate not in seen:
            seen.add(candidate)
            yield rule, ops, candidate

def reduced_bar(q):
    return max(0.05, 0.5 * _wps_original_bar(q))
import time
_original_completion_score_edit=score_edit
_score_progress={'calls':0,'last':time.monotonic()}
def score_edit(text,e):
    _score_progress['calls']+=1
    now=time.monotonic()
    if now-_score_progress['last']>=15:
        print('  Contextual comparisons:',_score_progress['calls'],flush=True)
        _score_progress['last']=now
    return _original_completion_score_edit(text,e)


In [ ]:
evaluation=[(c,{'text':c['text']}) for c in prepared]
sentence_results={'ordinary':evaluation}
evaluation=sentence_results['ordinary']
def stable_rng(*parts):
    return np.random.default_rng(int(hashlib.sha256(json.dumps([SELECTION_SEED,*parts]).encode()).hexdigest()[:16],16))
def rates(text):
    st=stats_for(text)
    return {(m,d):text.count(sym)/(st['C'] if d=='relative' else st['S']) for m,sym in PMARKS.items() for d in DENOMS}
def outside(x,lo,hi): return max(lo-x,0.,x-hi)
training=features.loc[(features.dataset=='raid')&(features.split=='train')].copy()
genre_ref={}
needed=[(m,d) for m,ds in active.items() for d in ds]
for genre in sorted({c['domain'] for c,s in evaluation}):
    g=training.loc[training.domain==genre]
    hr=[rates(normalize(text_map[r])) for r in g.loc[g.label==0,'source_id']]
    ar=[rates(normalize(text_map[r])) for r in g.loc[g.label==1,'source_id']]
    assert len(hr)>=15 and len(ar)>=15
    for m,d in needed:
        h=np.array([r[m,d] for r in hr]); a=np.array([r[m,d] for r in ar])
        rg=stable_rng('genre',genre,m,d)
        sd=float(np.std([h[rg.integers(len(h),size=len(h))].mean() for _ in range(BOOTSTRAPS)],ddof=1))
        mu=float(h.mean()); direction=float(np.sign(a.mean()-mu))
        genre_ref[genre,m,d]=dict(mean=mu,sd=sd,lo=max(0,mu-ARTIFICIAL_SD_SCALE*sd),hi=mu+ARTIFICIAL_SD_SCALE*sd,direction=direction)

compound_examples={}
for genre in sorted({c['domain'] for c,s in evaluation}):
    forms=set()
    for sid in humans.loc[humans.domain==genre,'source_id']:
        forms.update(x.lower() for x in re.findall(r'\b[A-Za-z]+(?:-[A-Za-z]+)+\b',str(text_map[sid])))
    compound_examples[genre]=forms

def vertices(text,mark,genre):
    doc=nlp(text); out=set(); symbol=PMARKS[mark]
    def emit(a,b,r):
        old=text[a:b]
        if old==r or any(x in old+r for x in '.?!'):return
        if any(sym in old+r for m,sym in PMARKS.items() if m not in (mark,'comma')):return
        out.add((a,b,r))
    for hit in re.finditer(re.escape(symbol),text):
        a,b=hit.span()
        if (a and text[a-1].isdigit()) or (b<len(text) and text[b].isdigit()):continue
        if mark=='comma': emit(a,b,' ' if a and b<len(text) and text[a-1].isalnum() and text[b].isalnum() else '')
        elif mark=='hyphen' and a and b<len(text) and text[a-1].isalpha() and text[b].isalpha():emit(a,b,' ')
        else:
            while a and text[a-1] in ' \t':a-=1
            while b<len(text) and text[b] in ' \t':b+=1
            options={';':[', and ', ', but ', ', so ', ', '], ':':[', namely ', ', including ', ' including ', ', specifically ', ', which is ', ', which are ', ' such as ', ', ']}.get(symbol,[', ',' ', ', namely ', ', and ', ', which is ', ', which are '])
            for r in options:emit(a,b,r)
    for sent in doc.sents:
        if mark=='semicolon':
            for hit in re.finditer(r',\s*|\b(?:and|but|yet|so)\b',sent.text,re.I):
                a,b=sent.start_char+hit.start(),sent.start_char+hit.end()
                if independent(text[sent.start_char:a]) and independent(text[b:sent.end_char]):emit(a,b,'; ')
        if mark in ['colon','hyphen','en_dash','em_dash']:
            for hit in re.finditer(r'\b(?:namely|specifically|including|such as|that is|in other words)\b',sent.text,re.I):
                a,b=sent.start_char+hit.start(),sent.start_char+hit.end()
                if any(t.isalpha() for t in text[sent.start_char:a]) and any(t.isalpha() for t in text[b:sent.end_char]):emit(a,b,' '+symbol+' ')
        if mark=='comma':
            for t in sent:
                if t.dep_ not in {'advcl','appos','relcl','parataxis','conj','prep'}:continue
                subtree=list(t.subtree);a=min(x.idx for x in subtree);b=max(x.idx+len(x.text) for x in subtree)
                if a<sent.start_char or b>sent.end_char:continue
                if a>sent.start_char and text[a-1].isspace() and text[:a].rstrip()[-1:] not in ',;:—–-.?!':emit(a,a,', ')
                if b<sent.end_char and text[b:b+1].isspace() and t.dep_ in {'advcl','appos','relcl','prep'}:emit(b,b,',')
    if mark=='hyphen':
        toks=list(doc)
        for left,right in zip(toks,toks[1:]):
            a=left.idx+len(left.text);b=right.idx
            if left.is_alpha and right.is_alpha and text[a:b].isspace() and (left.text+'-'+right.text).lower() in compound_examples[genre]:emit(a,b,'-')
    for a,b,r in sorted(out):yield dict(start=a,end=b,replacement=r)

def mark_loss(text,p,mark):
    target=p['targets'][mark]
    if target is not None:return abs(text.count(PMARKS[mark])-target)
    st=stats_for(text); vals=rates(text); total=0.
    for d in active[mark]:
        lo,hi=p['local'][mark,d]
        exposure=p['initial']['C'] if d=='relative' else p['initial']['S']
        total+=outside(vals[mark,d],lo,hi)/max(hi-lo,1/exposure)
    return total

def enforce(p):
    text=p['text']; initial=p['initial'];c=p['case'];seen={text};edits=[];stops=[];done=[]
    for mark,ds in active.items():
        if not ds:continue
        while True:
            loss=mark_loss(text,p,mark)
            if loss<=1e-10:
                stops.append(dict(mark=mark,status='in_assigned_target',loss=loss));done.append(mark);break
            counters=Counter(); eligible=[]
            for e in vertices(text,mark,c['domain']):
                counters['proposed']+=1
                candidate=text[:e['start']]+e['replacement']+text[e['end']:]
                if candidate in seen:counters['visited']+=1;continue
                after=stats_for(candidate)
                if after['S']!=initial['S'] or any(candidate.count(x)!=p['text'].count(x) for x in '.?!'):
                    counters['boundary_change']+=1;continue
                if any(candidate.count(sym)!=text.count(sym) for m,sym in PMARKS.items() if m not in [mark,'comma']):
                    counters['other_nontrivial_change']+=1;continue
                if any(outside(after[m],c['bounds'][m]['low'],c['bounds'][m]['high'])>
                       outside(initial[m],c['bounds'][m]['low'],c['bounds'][m]['high'])+MEAN_DISTANCE_TOLERANCE[m]+1e-10 for m in SENT_METRICS):
                    counters['mean_control_regression']+=1;continue
                if any(mark_loss(candidate,p,m)>1e-10 for m in done):counters['completed_target_regression']+=1;continue
                newloss=mark_loss(candidate,p,mark)
                if newloss>=loss-1e-10:counters['no_target_improvement']+=1;continue
                score=score_edit(text,e)
                if score is None:counters['unscorable']+=1;continue
                eligible.append((score['mutability'],loss-newloss,candidate,e,score))
            stops.append(dict(mark=mark,status='search_iteration',loss=loss,**counters,eligible=len(eligible)))
            if not eligible:
                stops.append(dict(mark=mark,status='enumerated_candidates_exhausted',loss=loss));break
            eligible.sort(key=lambda r:(-r[0],-r[1],r[2]))
            mut,gain,candidate,e,score=eligible[0]
            edits.append(dict(mark=mark,before=text,after=candidate,loss_before=loss,loss_after=loss-gain,edit=e,**score))
            text=candidate;seen.add(text)
        print(c['case_id'],mark,'loss',round(mark_loss(text,p,mark),5),flush=True)
    assert all(text.count(x)==p['text'].count(x) for x in '.?!')
    assert stats_for(text)['S']==initial['S']
    assert all(text.count(PMARKS[m])==p['text'].count(PMARKS[m]) for m,ds in active.items() if not ds)
    return dict(text=text,edits=edits,stops=stops)


def cached_run(path, specification, fn):
    key = hashlib.sha256(
        json.dumps(specification, sort_keys=True, default=str).encode()
    ).hexdigest()
    if path.exists():
        payload = json.loads(path.read_text())
        assert payload["key"] == key, (
            f"Configuration changed: use a new output directory: {path}"
        )
        return payload["result"]
    result = fn()
    path.write_text(json.dumps(
        {"key": key, "result": result}, indent=2, ensure_ascii=False
    ))
    return result

def rebuild_targets(old_p, text):
    # Recompute exposure-dependent targets with the original RNG keys.
    p = copy.deepcopy(old_p)
    p["text"] = text
    p["initial"] = stats_for(text)
    p["targets"] = {}
    for mark, ds in active.items():
        if not ds:
            continue
        probabilities = []
        for denominator in ds:
            lo, hi = p["local"][mark, denominator]
            g = genre_ref[p["case"]["domain"], mark, denominator]
            trigger = (
                (g["direction"] > 0 and lo > g["hi"]) or
                (g["direction"] < 0 and hi < g["lo"])
            )
            distance = max(g["lo"] - hi, lo - g["hi"], 0) / max(
                g["sd"], abs(g["mean"]) * 1e-6, 1e-12
            )
            probabilities.append(
                BUMP_P_MIN + (1-BUMP_P_MIN)/(1+distance)
                if trigger else 0.
            )
        rng = stable_rng("target", p["case"]["case_id"], mark)
        bumped = rng.random() < max(probabilities)
        basis = "per_sentence" if "per_sentence" in ds else "relative"
        exposure = p["initial"]["S" if basis == "per_sentence" else "C"]
        expected = genre_ref[p["case"]["domain"], mark, basis]["mean"] * exposure
        p["targets"][mark] = (
            int(np.floor(expected) + (rng.random() < expected % 1))
            if bumped else None
        )
    return p

def terminal_swaps(text):
    # Only a standalone terminal punctuation token; skip initials,
    # abbreviations attached to tokens, ellipses and mixed punctuation.
    for sent in nlp(text).sents:
        tokens = [t for t in sent if not t.is_space]
        while tokens and tokens[-1].text in {'"', "'", "”", "’", ")", "]"}:
            tokens.pop()
        if not tokens or tokens[-1].text not in {".", "!"}:
            continue
        t = tokens[-1]
        a = t.idx
        if (a and text[a-1] in ".?!") or text[a+1:a+2] in {".", "?", "!"}:
            continue
        if t.text == "." and len(tokens) > 1:
            prev = tokens[-2]
            if prev.is_alpha and len(prev.text) == 1:
                continue
        yield dict(start=a, end=a+1,
                   replacement="." if t.text == "!" else "!")

def fix_exclamation(text, refs):
    # Local training-reference bands, with no held-out-human targeting.
    hr = []
    for sid in refs:
        t = normalize(text_map[sid])
        st = stats_for(t)
        hr.append([t.count("!") / st["C"], t.count("!") / st["S"]])
    bands = np.quantile(np.asarray(hr), [.2, .6], axis=0)
    initial = stats_for(text)

    def loss(t):
        st = stats_for(t)
        values = [t.count("!") / st["C"], t.count("!") / st["S"]]
        exposures = [initial["C"], initial["S"]]
        return sum(
            outside(v, bands[0, j], bands[1, j]) /
            max(bands[1, j] - bands[0, j], 1/exposures[j])
            for j, v in enumerate(values)
        )

    edits = []
    while loss(text) > 1e-10:
        candidates = []
        before_doc = nlp(text)
        sentence_words = [
            signature(s.text) for s in before_doc.sents
            if any(t.is_alpha for t in s)
        ]
        for e in terminal_swaps(text):
            candidate = text[:e["start"]] + e["replacement"] + text[e["end"]:]
            if signature(candidate) != signature(text):
                continue
            # Preserve the word membership of each sentence, not just S.
            after_sentences = [
                signature(s.text) for s in nlp(candidate).sents
                if any(t.is_alpha for t in s)
            ]
            if after_sentences != sentence_words:
                continue
            st = stats_for(candidate)
            if any(st[k] != initial[k] for k in ("W", "S", "C")):
                continue
            gain = loss(text) - loss(candidate)
            if gain <= 1e-10:
                continue
            score = score_edit(text, e)
            if score is None:
                continue
            # Same relaxed sentence-score policy; not the generation ratio.
            threshold = reduced_bar(loss(text))
            if score["mutability"] < threshold:
                continue
            candidates.append((score["mutability"], gain, candidate, e, score))
        if not candidates:
            break
        candidates.sort(key=lambda x: (-x[0], -x[1], x[2]))
        _, _, candidate, e, score = candidates[0]
        edits.append(dict(before=text, after=candidate, edit=e, **score))
        text = candidate
    return dict(text=text, edits=edits, loss=loss(text),
                bands=bands.tolist())
all_training_compounds = set()
for sid in humans.source_id:
    all_training_compounds.update(
        m.lower() for m in re.findall(
            r"\b[A-Za-z]+(?:-[A-Za-z]+)+\b", str(text_map[sid])
        )
    )

def number_strings(text):
    return re.findall(r"\d+(?:[./]\d+)*|[¼½¾⅓⅔⅛⅜⅝⅞]", text)

def extra_hyphen_vertices(text, genre):
    doc = nlp(text)

    # Compounds observed in training-human text.
    tokens = list(doc)
    for left, right in zip(tokens, tokens[1:]):
        a, b = left.idx + len(left.text), right.idx
        compound = (left.text + "-" + right.text).lower()
        if (
            left.is_alpha and right.is_alpha
            and text[a:b].isspace()
            and "\n" not in text[a:b]
            and compound in all_training_compounds
        ):
            yield dict(start=a, end=b, replacement="-",
                       proposal_type="training_compound")

    # Recipe markers immediately before quantity + unit.
    # Scores/ranges such as 6-2 do not match this rule.
    if genre == "recipes":
        quantity_unit = re.compile(
            r"-[ \t]*(?=\d+(?:[./]\d+)?(?:[ \t]+[¼½¾⅓⅔⅛⅜⅝⅞])?"
            r"[ \t]+(?:cups?|tablespoons?|teaspoons?|tbsp|tsp|"
            r"grams?|kilograms?|ounces?|pounds?|ml|liters?|litres?)\b)",
            re.I,
        )
        for hit in quantity_unit.finditer(text):
            a, b = hit.span()
            if a and (text[a-1].isdigit() or text[a-1] in "¼½¾⅓⅔⅛⅜⅝⅞"):
                continue
            # A space removes the marker without joining adjacent words.
            yield dict(start=a, end=b, replacement=" ",
                       proposal_type="recipe_list_marker")

def repair_hyphens(text, p):
    initial = stats_for(text)
    seen = {text}
    edits, stops = [], []

    while mark_loss(text, p, "hyphen") > 1e-10:
        loss = mark_loss(text, p, "hyphen")
        candidates, counts = [], {}
        vertices_seen = set()

        proposals_here = [
            {**e, "proposal_type": "original"}
            for e in vertices(text, "hyphen", p["case"]["domain"])
        ]
        proposals_here += list(extra_hyphen_vertices(
            text, p["case"]["domain"]
        ))

        for e in proposals_here:
            key = (e["start"], e["end"], e["replacement"])
            if key in vertices_seen:
                continue
            vertices_seen.add(key)
            candidate = text[:e["start"]] + e["replacement"] + text[e["end"]:]

            def reject(reason):
                counts[reason] = counts.get(reason, 0) + 1

            if candidate in seen:
                reject("visited")
                continue
            if number_strings(candidate) != number_strings(text):
                reject("numeric_change")
                continue
            st = stats_for(candidate)
            if st["S"] != initial["S"]:
                reject("sentence_count_change")
                continue
            if any(candidate.count(x) != text.count(x) for x in ".?!"):
                reject("terminal_punctuation_change")
                continue
            if any(
                candidate.count(sym) != text.count(sym)
                for m, sym in PMARKS.items()
                if m != "hyphen"
            ):
                reject("other_punctuation_change")
                continue
            if any(
                outside(st[m], p["case"]["bounds"][m]["low"],
                        p["case"]["bounds"][m]["high"])
                > outside(initial[m], p["case"]["bounds"][m]["low"],
                          p["case"]["bounds"][m]["high"])
                + MEAN_DISTANCE_TOLERANCE[m] + 1e-10
                for m in SENT_METRICS
            ):
                reject("mean_regression")
                continue

            # Preserve other active punctuation losses.
            if any(
                mark_loss(candidate, p, m) > mark_loss(text, p, m) + 1e-10
                for m, ds in active.items() if ds and m != "hyphen"
            ):
                reject("other_target_regression")
                continue
            gain = loss - mark_loss(candidate, p, "hyphen")
            if gain <= 1e-10:
                reject("no_improvement")
                continue
            score = score_edit(text, e)
            if score is None:
                reject("unscorable")
                continue
            candidates.append((score["mutability"], gain, candidate, e, score))

        stops.append(dict(loss=loss, **counts, eligible=len(candidates)))
        if not candidates:
            break
        candidates.sort(key=lambda x: (-x[0], -x[1], x[2]))
        _, _, candidate, e, score = candidates[0]
        edits.append(dict(before=text, after=candidate, edit=e, **score))
        text = candidate
        seen.add(text)

    return dict(text=text, edits=edits, stops=stops,
                final_loss=mark_loss(text, p, "hyphen"))



In [ ]:
def stage_cached(cid,name,inputs,fn):
    folder=COMPLETION/cid;folder.mkdir(exist_ok=True)
    path=folder/(name+'.json')
    key=hashlib.sha256(json.dumps(dict(contract=CONTRACT_HASH,inputs=inputs),sort_keys=True,default=str).encode()).hexdigest()
    if path.exists():
        value=json.loads(path.read_text());assert value['key']==key,f'Changed checkpoint: {path}'
        return value['result']
    print(cid,name,'starting',flush=True)
    result=fn();atomic(path,dict(key=key,result=result));return result

def punctuation_spec(c,text):
    local={}
    reference_rates=[rates(normalize(text_map[sid])) for sid in c['reference_ids']]
    for mark,ds in active.items():
        for d in ds:local[mark,d]=tuple(map(float,np.quantile([v[mark,d] for v in reference_rates],[.2,.6])))
    return rebuild_targets(dict(case=c,local=local),text)

input_frame=pd.read_csv(INPUT,keep_default_na=False).set_index('case_id')
completion_rows=[];reviews=[]
for i,c in enumerate(prepared,1):
    cid=c['case_id'];print(f'Completion {i}/{len(prepared)}: {cid}',flush=True)
    # Original sentence pass, then the same conditional WPS repair used in C02s.
    proposals,bar=_wps_original_proposals,_wps_original_bar
    first=stage_cached(cid,'sentence_original',c,lambda:run_case(c))
    if distances(first['final'],c['bounds'])[0]>1e-9:
        proposals,bar=expanded_proposals,reduced_bar
        second=stage_cached(cid,'sentence_repair',dict(case=c,text=first['text']),lambda:run_case({**c,'text':first['text']}))
    else:second=first
    proposals,bar=_wps_original_proposals,_wps_original_bar
    p=punctuation_spec(c,second['text'])
    punct=stage_cached(cid,'punctuation',dict(case=c,text=second['text']),lambda:enforce(p))
    ex=stage_cached(cid,'exclamation',dict(text=punct['text'],refs=c['reference_ids']),lambda:fix_exclamation(punct['text'],c['reference_ids']))
    hy=stage_cached(cid,'hyphen',dict(case=c,text=ex['text'],sentence_text=second['text']),lambda:repair_hyphens(ex['text'],p))
    input_frame.loc[cid,['text','pipeline_status','provenance']]=[hy['text'],'complete',str((COMPLETION/cid/'hyphen.json').relative_to(ROOT))]
    tmp=INPUT.with_suffix('.tmp');input_frame.reset_index().to_csv(tmp,index=False);os.replace(tmp,INPUT)
    completion_rows.append(dict(case_id=cid,sentence_status=second['status'],sentence_WPS_distance=float(distances(second['final'],c['bounds'])[0]),
        sentence_CPS_distance=float(distances(second['final'],c['bounds'])[1]),punctuation_edits=len(punct['edits']),
        exclamation_edits=len(ex['edits']),hyphen_edits=len(hy['edits']),hyphen_loss=hy['final_loss']))
    reviews.append(dict(case_id=cid,before=c['text'],after=hy['text'],fidelity_review='',grammar_review='',notes=''))
    print(cid,'saved full pipeline output',flush=True)
pd.DataFrame(completion_rows).to_csv(COMPLETION/'outcomes.csv',index=False)
if not (COMPLETION/'manual_review.csv').exists():pd.DataFrame(reviews).to_csv(COMPLETION/'manual_review.csv',index=False)
for cid,text in known.items():assert input_frame.loc[cid,'text']==text
assert input_frame.pipeline_status.eq('complete').all() and input_frame.text.str.strip().ne('').all()
print('All 54 final outputs available. Existing 27 unchanged. Continue to the adaptation cells.')
# Free GPT-2 before CPU forest fitting; no more language-model scoring needed.
del model,tokenizer;gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()


In [ ]:
merged=pd.read_csv(INPUT,keep_default_na=False)
assert len(merged)==54 and merged.case_id.is_unique
assert merged.pipeline_status.eq('complete').all() and merged.text.str.strip().ne('').all()
new_ids=[c['case_id'] for c in prepared]
merged.loc[merged.case_id.isin(new_ids)].to_csv(COMPLETION/'completed_27_texts.csv',index=False)
merged.to_csv(COMPLETION/'all_54_texts.csv',index=False)
atomic(COMPLETION/'handoff.json',dict(final_input=str(INPUT.relative_to(ROOT)),sha256=sha(INPUT),
    total=54,preserved=27,completed=27,reference_policy='reference_policy.csv',quality_review='manual_review.csv'))
report=pd.DataFrame(completion_rows)
display(report)
fig,ax=plt.subplots(figsize=(10,4))
report.set_index('case_id')[['punctuation_edits','exclamation_edits','hyphen_edits']].plot.bar(stacked=True,ax=ax)
ax.set_ylabel('Accepted punctuation operations');ax.set_title('Completion of 27 previously missing final texts')
fig.tight_layout()
for ext in ['png','pdf']:_export_figure(fig,COMPLETION/f'completion_operations.{ext}',dpi=300,bbox_inches='tight')
plt.show()
print('DONE: all 54 inputs saved to',INPUT)
print('Return to C02w and rerun its first two code cells. The completeness gate should now pass.')
